In [0]:
spark.sql("USE CATALOG workspace")

from pyspark.sql.functions import (
    col, trim, lit, when, current_timestamp, initcap
)

#read from bronze and silver reference tables
bronze_df     = spark.table("bronze.skills")
silver_emp_df = spark.table("silver.employees").select("employee_id")

print(f"Bronze skills:      {bronze_df.count()}")
print(f"Silver employees:   {silver_emp_df.count()}")
bronze_df.printSchema()

In [0]:
# Apply cleaning transformations 
valid_emp_ids = [row.employee_id for row in silver_emp_df.collect()]

VALID_PROFICIENCIES = ["Beginner", "Intermediate", "Advanced", "Expert"]

cleaned_df = (
    bronze_df
    # Trim whitespace
    .withColumn("skill",       trim(col("skill")))
    .withColumn("proficiency", initcap(trim(col("proficiency"))))

    # Validate proficiency values
    .withColumn("proficiency_valid", when(
        col("proficiency").isin(VALID_PROFICIENCIES),
        lit(True)).otherwise(lit(False)))

    # Validate employee IDs exist in silver
    .withColumn("employee_valid", when(
        col("employee_id").isin(valid_emp_ids),
        lit(True)).otherwise(lit(False)))

    # Flag nulls in critical fields
    .withColumn("has_nulls", when(
        col("employee_id").isNull() |
        col("skill").isNull() |
        col("proficiency").isNull(),
        lit(True)).otherwise(lit(False)))

    # Add audit columns
    .withColumn("silver_loaded_at", current_timestamp())
    .withColumn("source_table",     lit("bronze.skills"))
)

# Deduplicate on employee_id + skill keeping most recent assessment
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("employee_id", "skill").orderBy(col("last_assessed").desc())
cleaned_df = (
    cleaned_df
    .withColumn("row_num", row_number().over(window))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

print(f"Silver row count:          {cleaned_df.count()}")
print(f"Invalid proficiencies:     {cleaned_df.filter(col('proficiency_valid') == False).count()}")
print(f"Invalid employee IDs:      {cleaned_df.filter(col('employee_valid') == False).count()}")
print(f"Rows with nulls:           {cleaned_df.filter(col('has_nulls') == True).count()}")
cleaned_df.show(5)

In [0]:
(
    cleaned_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.skills")
)

print(f"Written to silver.skills: {spark.table('silver.skills').count()} rows")